In [30]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Users/javier.escobar/code/github/skforecast
0.23.0


In [31]:
import re
import pytest
import numpy as np
import pandas as pd
from sklearn.exceptions import NotFittedError
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from skforecast.preprocessing import RollingFeatures
from skforecast.preprocessing import TimeSeriesDifferentiator
from skforecast.direct import ForecasterDirect

# Fixtures
from skforecast.direct.tests.tests_forecaster_direct.fixtures_forecaster_direct import y as y_categorical
from skforecast.direct.tests.tests_forecaster_direct.fixtures_forecaster_direct import exog as exog_categorical
from skforecast.direct.tests.tests_forecaster_direct.fixtures_forecaster_direct import exog_predict as exog_predict_categorical
from skforecast.direct.tests.tests_forecaster_direct.fixtures_forecaster_direct import data  # to test results when using differentiation

In [32]:
from skforecast.preprocessing import CalendarFeatures

In [33]:
# Libraries
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from skforecast.datasets import fetch_dataset
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.plot import plot_prediction_intervals, set_dark_theme

In [34]:
X_idx = pd.date_range(start='2020-01-01', end='2026-12-31', freq='D')

In [35]:
calendar = CalendarFeatures(encoding=None)
X_train = calendar.fit_transform(X_idx)
X_train.head(10)

,year,month,week,day_of_week,day_of_month,day_of_year,weekend,hour,minute,second,quarter
2020-01-01,2020,1,1,2,1,1,0,0,0,0,1
2020-01-02,2020,1,1,3,2,2,0,0,0,0,1
2020-01-03,2020,1,1,4,3,3,0,0,0,0,1
2020-01-04,2020,1,1,5,4,4,1,0,0,0,1
2020-01-05,2020,1,1,6,5,5,1,0,0,0,1
2020-01-06,2020,1,2,0,6,6,0,0,0,0,1
2020-01-07,2020,1,2,1,7,7,0,0,0,0,1
2020-01-08,2020,1,2,2,8,8,0,0,0,0,1
2020-01-09,2020,1,2,3,9,9,0,0,0,0,1
2020-01-10,2020,1,2,4,10,10,0,0,0,0,1


In [36]:
calendar.features

['year',
 'month',
 'week',
 'day_of_week',
 'day_of_month',
 'day_of_year',
 'weekend',
 'hour',
 'minute',
 'second',
 'quarter']

In [37]:
cols = ['year',
 'month',
 'week',
 'day_of_week',
 'day_of_month',
 'day_of_year',
 'weekend',
 'hour',
 'minute',
 'second',
 'quarter']

X_raw = pd.DataFrame(index=X_idx)

X_raw['year'] = X_raw.index.year
X_raw['month'] = X_raw.index.month
X_raw['week'] = X_raw.index.isocalendar().week
X_raw['day_of_week'] = X_raw.index.dayofweek
X_raw['day_of_month'] = X_raw.index.day
X_raw['day_of_year'] = X_raw.index.dayofyear
X_raw['weekend'] = X_raw.index.dayofweek >= 5
X_raw['hour'] = X_raw.index.hour
X_raw['minute'] = X_raw.index.minute
X_raw['second'] = X_raw.index.second
X_raw['quarter'] = X_raw.index.quarter

In [38]:
pd.testing.assert_frame_equal(X_train, X_raw, check_dtype=False)

## Encoding

In [39]:
# Libraries
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt
from skforecast.datasets import fetch_dataset
from feature_engine.datetime import DatetimeFeatures
from feature_engine.creation import CyclicalFeatures

In [40]:
calendar = CalendarFeatures(encoding="cyclical")
X_train = calendar.fit_transform(X_idx)
X_train.head(2)

,year,weekend,month_sin,month_cos,week_sin,week_cos,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos,minute_sin,minute_cos,second_sin,second_cos,quarter_sin,quarter_cos
2020-01-01,2020,0,0.5,0.866025,0.118273,0.992981,0.974928,-0.222521,0.201299,0.979530,0.017166,0.999853,0.0,1.0,0.0,1.0,0.0,1.0,1.0,6.123234e-17
2020-01-02,2020,0,0.5,0.866025,0.118273,0.992981,0.433884,-0.900969,0.394356,0.918958,0.034328,0.999411,0.0,1.0,0.0,1.0,0.0,1.0,1.0,6.123234e-17


In [45]:
X_raw = pd.DataFrame(index=X_idx)

X_raw['year'] = X_raw.index.year
X_raw['month'] = X_raw.index.month
X_raw['week'] = X_raw.index.isocalendar().week
X_raw['day_of_week'] = X_raw.index.dayofweek
X_raw['day_of_month'] = X_raw.index.day
X_raw['day_of_year'] = X_raw.index.dayofyear
X_raw['weekend'] = X_raw.index.dayofweek >= 5
X_raw['hour'] = X_raw.index.hour
X_raw['minute'] = X_raw.index.minute
X_raw['second'] = X_raw.index.second
X_raw['quarter'] = X_raw.index.quarter

In [46]:
# Cyclical encoding
# ==============================================================================
features_to_encode = [
    "quarter",
    "month",
    "week",
    "day_of_week",
    "day_of_month",
    "day_of_year",
    "hour",
    "minute",
    "second",
]
max_values = {
    "quarter": 4,
    "month": 12,
    "week": 53,
    "day_of_week": 7,
    "day_of_month": 31,
    "day_of_year": 366,
    "hour": 24,
    "minute": 60,
    "second": 60,
}
cyclical_encoder = CyclicalFeatures(
    variables     = features_to_encode,
    max_values    = max_values,
    drop_original = True
)

X_raw = cyclical_encoder.fit_transform(X_raw)
X_raw.head(3)

,year,weekend,quarter_sin,quarter_cos,month_sin,month_cos,week_sin,week_cos,day_of_week_sin,day_of_week_cos,day_of_month_sin,day_of_month_cos,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos,minute_sin,minute_cos,second_sin,second_cos
2020-01-01,2020,False,1.0,6.123234e-17,0.5,0.866025,0.118273,0.992981,0.974928,-0.222521,0.201299,0.979530,0.017166,0.999853,0.0,1.0,0.0,1.0,0.0,1.0
2020-01-02,2020,False,1.0,6.123234e-17,0.5,0.866025,0.118273,0.992981,0.433884,-0.900969,0.394356,0.918958,0.034328,0.999411,0.0,1.0,0.0,1.0,0.0,1.0
2020-01-03,2020,False,1.0,6.123234e-17,0.5,0.866025,0.118273,0.992981,-0.433884,-0.900969,0.571268,0.820763,0.051479,0.998674,0.0,1.0,0.0,1.0,0.0,1.0


In [47]:
pd.testing.assert_frame_equal(X_train, X_raw[X_train.columns], check_dtype=False)